# Customers and Orders - Technical Challenge
This notebook contains the solutions for Exercise 1 (Customers) and Exercise 2 (Orders).

In [ ]:
# Download data files
import requests
import os

os.makedirs('sample_data', exist_ok=True)

customers_url = 'https://raw.githubusercontent.com/anyoneai/notebooks/main/customers_and_orders/data/customers.csv'
orders_url = 'https://raw.githubusercontent.com/anyoneai/notebooks/main/customers_and_orders/data/orders.csv'

with open('sample_data/customers.csv', 'wb') as f:
    f.write(requests.get(customers_url).content)

with open('sample_data/orders.csv', 'wb') as f:
    f.write(requests.get(orders_url).content)

print('Files downloaded successfully!')

---
# Exercise 1: Processing Customers Data

## Q1 – How many rows does the customers.csv file have?

In [ ]:
with open('sample_data/customers.csv', 'r') as f:
    lines = f.readlines()

# Total rows including header
total_rows = len(lines)
# Data rows (excluding header)
data_rows = total_rows - 1

print(f'Total rows (including header): {total_rows}')
print(f'Data rows (excluding header): {data_rows}')

## Q2 – How many columns does the customers.csv file have?

In [ ]:
with open('sample_data/customers.csv', 'r') as f:
    header = f.readline()

columns = header.strip().split(',')
print(f'Number of columns: {len(columns)}')
print(f'Column names: {columns}')

## Q3 – Load customers.csv into a list of dictionaries

In [ ]:
import csv

customers = []
with open('sample_data/customers.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        customers.append(dict(row))

print(f'Number of customers loaded: {len(customers)}')
print('First customer:', customers[0])

## Q4 – How many customers have a last name that ends with the letter 's'?

In [ ]:
count_last_name_s = 0
for customer in customers:
    last_name = customer.get('last_name', '').strip()
    if last_name.lower().endswith('s'):
        count_last_name_s += 1

print(f'Customers with last name ending in s: {count_last_name_s}')

## Q5 – What is the most common "state" customers are from?

In [ ]:
state_count = {}
for customer in customers:
    state = customer.get('state', '').strip()
    if state:
        state_count[state] = state_count.get(state, 0) + 1

most_common_state = max(state_count, key=state_count.get)
print(f'Most common state: {most_common_state} ({state_count[most_common_state]} customers)')

## Q6 – What is the gender of the oldest customer?

In [ ]:
from datetime import datetime

oldest_customer = None
oldest_date = None

for customer in customers:
    dob_str = customer.get('date_of_birth', '').strip()
    if not dob_str:
        continue
    try:
        dob = datetime.strptime(dob_str, '%Y-%m-%d')
    except ValueError:
        try:
            dob = datetime.strptime(dob_str, '%m/%d/%Y')
        except ValueError:
            continue

    if oldest_date is None or dob < oldest_date:
        oldest_date = dob
        oldest_customer = customer

print(f'Oldest customer: {oldest_customer}')
print(f'Date of birth: {oldest_date}')
print(f'Gender: {oldest_customer.get("gender", "N/A")}')

## Q7 – Is there duplicate data in the customers file?

In [ ]:
# Check for duplicate customer IDs
customer_ids = [c.get('customer_id', c.get('id', '')).strip() for c in customers]
unique_ids = set(customer_ids)

has_duplicates = len(customer_ids) != len(unique_ids)
print(f'Total records: {len(customer_ids)}')
print(f'Unique IDs: {len(unique_ids)}')
print(f'Has duplicate data: {has_duplicates}')

if has_duplicates:
    print(f'Number of duplicates: {len(customer_ids) - len(unique_ids)}')

---
# Exercise 2: Processing Orders Data

## Load Orders data

In [ ]:
import csv

orders = []
with open('sample_data/orders.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        orders.append(dict(row))

print(f'Number of order rows loaded: {len(orders)}')
print('First row:', orders[0])

## Q1 – Orders for customer with id = 33 (how many distinct orders?)

In [ ]:
customer_id_target = '33'

orders_for_customer = [o for o in orders if o.get('customer_id', '').strip() == customer_id_target]
distinct_order_ids = set(o.get('order_id', '').strip() for o in orders_for_customer)

print(f'Rows for customer {customer_id_target}: {len(orders_for_customer)}')
print(f'Distinct orders for customer {customer_id_target}: {len(distinct_order_ids)}')

## Q2 – How many orders contain more than one item?

In [ ]:
from collections import Counter

order_item_count = Counter()
for o in orders:
    order_id = o.get('order_id', '').strip()
    if order_id:
        order_item_count[order_id] += 1

orders_with_multiple_items = sum(1 for count in order_item_count.values() if count > 1)
print(f'Orders with more than one item: {orders_with_multiple_items}')

## Q3 – What is the total revenue from all orders?

In [ ]:
total_revenue = 0.0
for o in orders:
    try:
        price = float(o.get('total_price', o.get('price', 0) or 0))
        total_revenue += price
    except (ValueError, TypeError):
        pass

print(f'Total revenue: ${total_revenue:,.2f}')

## Q4 – What is the average revenue per order?

In [ ]:
# Sum revenue per unique order_id first
order_revenue = {}
for o in orders:
    order_id = o.get('order_id', '').strip()
    try:
        price = float(o.get('total_price', o.get('price', 0) or 0))
    except (ValueError, TypeError):
        price = 0.0
    order_revenue[order_id] = order_revenue.get(order_id, 0.0) + price

if order_revenue:
    avg_revenue = sum(order_revenue.values()) / len(order_revenue)
    print(f'Number of unique orders: {len(order_revenue)}')
    print(f'Average revenue per order: ${avg_revenue:,.2f}')

## Q5 – Which customer made the most orders?

In [ ]:
# Count distinct orders per customer
customer_orders = {}
for o in orders:
    cid = o.get('customer_id', '').strip()
    oid = o.get('order_id', '').strip()
    if cid not in customer_orders:
        customer_orders[cid] = set()
    customer_orders[cid].add(oid)

customer_order_counts = {cid: len(oids) for cid, oids in customer_orders.items()}
top_customer = max(customer_order_counts, key=customer_order_counts.get)

print(f'Customer with most orders: ID={top_customer}, Orders={customer_order_counts[top_customer]}')

## Q6 – What is the most expensive order ever placed?

In [ ]:
# Sum prices per order to get total cost of each order
order_totals = {}
for o in orders:
    order_id = o.get('order_id', '').strip()
    try:
        price = float(o.get('total_price', o.get('price', 0) or 0))
    except (ValueError, TypeError):
        price = 0.0
    order_totals[order_id] = order_totals.get(order_id, 0.0) + price

most_expensive_order = max(order_totals, key=order_totals.get)
print(f'Most expensive order: ID={most_expensive_order}, Total=${order_totals[most_expensive_order]:,.2f}')

## Q7 – What is the revenue generated by each customer? (top 5)

In [ ]:
customer_revenue = {}
for o in orders:
    cid = o.get('customer_id', '').strip()
    try:
        price = float(o.get('total_price', o.get('price', 0) or 0))
    except (ValueError, TypeError):
        price = 0.0
    customer_revenue[cid] = customer_revenue.get(cid, 0.0) + price

sorted_customers = sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)
print('Top 5 customers by revenue:')
for cid, rev in sorted_customers[:5]:
    print(f'  Customer ID: {cid} -> ${rev:,.2f}')